In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 

In [89]:
df = pd.read_csv('../df_merged.csv',index_col=0)
rank_map = {1:6,2:5,3:4,4:3,5:2,6:1}
df_y = df['着']#.map(rank_map).astype(int)
df_x = df.drop(['名前','着'],axis=1)
x_train, x_vali, y_train, y_vali = train_test_split(df_x, df_y, test_size=0.2, shuffle=False)

train_group = x_train['RaceID'].value_counts(sort = False)
val_group = x_vali['RaceID'].value_counts(sort = False)

In [83]:

lgbm_params =  {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective': 'lambdarank', #←ここでランキング学習と指定！
    'metric': 'ndcg',   # for lambdarank
    'ndcg_eval_at': [1,2,3],  # 3連単を予測したい
    'max_position': 6,  # 競艇は6位までしかない
    'learning_rate': 0.01, 
    'group_column': 13,
    'min_data': 1,
    'min_data_in_bin': 1,
    #'num_leaves': 31,
   #'max_depth':35,
}

lgtrain = lgb.Dataset(x_train, y_train,  group=train_group)
lgvalid = lgb.Dataset(x_vali, y_vali,group=val_group)

lgb_clf = lgb.train(
    lgbm_params,
    lgtrain,
    num_boost_round=10000,
    valid_sets=[lgtrain, lgvalid],
    valid_names=['train','valid'],
    early_stopping_rounds=1000,
    verbose_eval=100
)

/home/tojo/.local/lib/python3.8/site-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/home/tojo/.local/lib/python3.8/site-packages/lightgbm/engine.py:240: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040268 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6487
[LightGBM] [Info] Number of data points in the train set: 211960, number of used features: 134
[LightGBM] [Warning] Unknown parameter: max_position
Training until validation scores don't improve for 1000 rounds
[100]	train's ndcg@1: 0.607202	train's ndcg@2: 0.663629	train's ndcg@3: 0.724217	valid's ndcg@1: 0.592077	valid's ndcg@2: 0.653355	valid's ndcg@3: 0.71481
[200]	train's ndcg@1: 0.616767	train's ndcg@2: 0.671206	train's ndcg@3: 0.730027	valid's ndcg@1: 0.594915	valid's ndcg@2: 0.655782	valid's ndcg@3: 0.717655
[300]	train's ndcg@1: 0.624485	train's ndcg@2: 0.677036	train's ndcg@3: 0.734882	valid's ndcg@1: 0.596551	val

In [84]:
y_pred_model = lgb_clf.predict(x_vali ,num_iteration=lgb_clf.best_iteration)

In [85]:
y_pred_model[:12]


array([-8.61088263e-01, -2.63534427e-01, -1.62796676e-03, -8.01978198e-01,
       -1.01099511e+00,  1.86667496e+00, -1.14675671e+00,  8.15603630e-01,
       -1.45533116e+00,  3.36595112e-01,  1.19643067e-01, -1.14282515e-01])

In [87]:
y_vali[:12]

211960    5
211961    3
211962    4
211963    6
211964    2
211965    1
211966    6
211967    1
211968    5
211969    4
211970    3
211971    2
Name: 着, dtype: int64

In [90]:
#Validデータ的中率の算出
j = 0
solo_count = 0
doub_count = 0
tri_count = 0
for i in val_group:
    result = y_pred_model[j:j+i]
    ans = y_vali[j:j+i].reset_index()

    result1st = np.argmin(result)
    if len(np.where(result==sorted(result)[1])[0])>1:
        result2nd = np.where(result==sorted(result)[1])[0][0]
        result3rd = np.where(result==sorted(result)[1])[0][1]
    else:
        if i > 1:
            result2nd = np.where(result==sorted(result)[1])[0][0]
        if i > 2:
            result3rd = np.where(result==sorted(result)[2])[0][0]

    ans1st = int(ans[ans["着"]==1].index.values)
    if len(ans[ans["着"]==2].index.values)>1:
        ans2nd = int(ans[ans["着"]==2].index.values[0])
        ans3rd = int(ans[ans["着"]==2].index.values[1])
    else:
        if i > 1:
            ans2nd = int(ans[ans["着"]==2].index.values[0])
        if i > 2:
            ans3rd = int(ans[ans["着"]==3].index.values[0])

    if ans1st==result1st:
        #print(ans1st,result1st)
        solo_count = solo_count+1

    if i > 1:
        if (ans1st==result1st)&(ans2nd==result2nd):
            doub_count = doub_count+1

    if i > 2:
        if (ans1st==result1st)&(ans2nd==result2nd)&(ans3rd==result3rd):
            tri_count = tri_count+1 
    j=j+i

print("単勝的中率：",round(solo_count/len(val_group)*100,2),"%")
print("２連単的中率：",round(doub_count/len(val_group)*100,2),"%")
print("３連単的中率：",round(tri_count/len(val_group)*100,2),"%")

単勝的中率： 55.8 %
２連単的中率： 22.1 %
３連単的中率： 9.16 %


In [ ]:
TARGET = '220830'
df_test = pd.DataFrame()
df_test = mkdf(TARGET,df_test, test=True)
df_test = std(lencode(df_test))
#df_test = df_test[df_test['場所'] == 13]
test_x = df_test#.drop(['RaceID'],axis=1)

test_group = mkgrp(test_x)
test_x = test_x#.drop(['R'],axis=1)
y_test_pred = lgb_clf.predict(test_x,group=test_group, num_iteration=lgb_clf.best_iteration)
y_test_pred = pd.DataFrame(y_test_pred)
y_pred = test_x.reset_index()
y_pred['pred'] = y_test_pred
import numpy as np
from scipy.stats import rankdata
l = y_pred.shape[0]//6
y_pred['pred_rank'] = 0
for i in range(l):
    p =[y_pred['pred'][i*6+ t] for t in range(6)]
    rp = rankdata(np.array(p))
    for j in range(6):
        y_pred['pred_rank'][i*6 + j] = rp[j]

y_pred.to_csv('../prediction/pred_'+TARGET+'.csv')
places = y_pred['場所'].unique()
place_list = {
            '桐生':'1',
            '戸田':'02',
            '江戸川':'03',
            '平和島':'04',
            '多摩川':'05',
            '浜名湖':'06',
            '蒲郡':'07',
            '常滑':'08',
            '津':'09',
            '三国':'10',
            'びわこ':'11',
            '住之江':'12',
            '尼崎':'13',
            '鳴門':'14',
            '丸亀':'15',
            '児島':'16',
            '宮島':'17',
            '徳山':'18',
            '下関':'19',
            '若松':'20',
            '芦屋':'21',
            '福岡':'22',
            '唐津':'23',
            '大村':'24'}

def get_keys_from_value(d, val):
    return [k for k, v in d.items() if v == val]

from os import makedirs
for p in places:
    if p < 10:
        v = '0' + str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    else:
        v = str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    makedirs('../prediction/' + TARGET + '/',exist_ok=True)
    with open("../prediction/"+TARGET+"/"+ TARGET+"_"+ PLACE +".txt", "w"):
        pass
    f = open("../prediction/"+TARGET+"/"+ TARGET+"_"+ PLACE +".txt", "w")
    print(PLACE,file=f)
    print(PLACE)
    y_predp = y_pred[y_pred['場所'] == p]
    pl = y_predp.shape[0]//6
    
    print('-----------------------',file=f)
    for i in range(pl):
        
        print('第'+str(i+1)+'R',file=f)
        race_pd = y_predp.iloc[i*6:i*6+6,:]
        first = race_pd[race_pd['pred_rank'] == 1]['艇番'].iloc[0]
        second = race_pd[race_pd['pred_rank'] == 2]['艇番'].iloc[0]
        third = race_pd[race_pd['pred_rank'] == 3]['艇番'].iloc[0]
        print(str(first)+'-'+str(second) +'-' + str(third),file=f)
        print(str(first)+'-'+str(second) +'-' + str(third))
    print('-----------------------\n',file=f)
    print('-----------------------\n')
    f.close()

NameError: name 'pd' is not defined